# Retail Data Warehouse – Exploratory Data Analysis

This notebook performs an in‑depth EDA on the `retail_dw` database.
**Schema**: star/snowflake with dimensions `customer`, `date`, `product`, `store`, `supplier` and facts `transaction`, `inventory`.

**Goals**:
- Understand data quality, distributions, relationships
- Identify sales drivers, profitable products, store/customer segments
- Detect inventory issues (stockouts, reorder patterns)
- Provide actionable business insights

## 1. Setup & Configuration

In [2]:
# Data handling
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Database connection
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

# Display options
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.2f}'.format)

# Visual style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Database connection – adjust credentials
# Use environment variables for security in production
DB_HOST = "localhost"
DB_USER = "root"
DB_PASSWORD = "S108108w"
DB_NAME = "retail_dw"

connection_string = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}"
try:
    engine = create_engine(connection_string)
    with engine.connect() as conn:
        print("✅ Connection successful")
except Exception as e:
    print("❌ Connection failed:", e)

## 2. Helper Functions
Generic function to load any table with a quick preview.

In [ ]:
def load_table(table_name, limit=None):
    query = f"SELECT * FROM {table_name}"
    if limit:
        query += f" LIMIT {limit}"
    return pd.read_sql(query, engine)

def basic_info(df, name):
    print(f"\n===== {name} =====")
    print(f"Shape: {df.shape}")
    print("Missing values:\n", df.isnull().sum())
    print("\nFirst 2 rows:")
    display(df.head(2))

## 3. Dimension Tables Exploration

In [ ]:
# Load all dimension tables (may be large, but retail dimensions are typically small/medium)
dim_customer = load_table('dim_customer')
dim_date = load_table('dim_date')
dim_product = load_table('dim_product')
dim_store = load_table('dim_store')
dim_supplier = load_table('dim_supplier')

basic_info(dim_customer, 'dim_customer')
basic_info(dim_date, 'dim_date')
basic_info(dim_product, 'dim_product')
basic_info(dim_store, 'dim_store')
basic_info(dim_supplier, 'dim_supplier')

### 3.1 Customer Demographics & Behaviour

In [ ]:
# Distribution of key customer attributes
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

dim_customer['gender'].value_counts().plot(kind='bar', ax=axes[0,0], title='Gender')
dim_customer['age_group'].value_counts().plot(kind='bar', ax=axes[0,1], title='Age Group')
dim_customer['city_tier'].value_counts().sort_index().plot(kind='bar', ax=axes[1,0], title='City Tier')
dim_customer['loyalty_member'].value_counts().plot(kind='bar', ax=axes[1,1], title='Loyalty Member')

plt.tight_layout()
plt.show()

# Avg monthly spend & visit frequency
print("Avg Monthly Spend (INR):")
print(dim_customer['avg_monthly_spend_inr'].describe())
print("\nVisit Frequency (per month):")
print(dim_customer['visit_frequency_per_month'].describe())

### 3.2 Product Assortment

In [ ]:
# Top categories by count, MRP range, margin profile
print("Number of unique categories:", dim_product['category'].nunique())
print("\nCategory counts:")
print(dim_product['category'].value_counts().head(10))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
dim_product['mrp'].hist(bins=50, ax=axes[0])
axes[0].set_title('MRP Distribution')
dim_product['gross_margin_pct'].hist(bins=30, ax=axes[1])
axes[1].set_title('Gross Margin % Distribution')
plt.show()

### 3.3 Store Footprint

In [ ]:
# Store types, cluster zones, and size
print("Store type distribution:")
print(dim_store['store_type'].value_counts())
print("\nCluster zones:")
print(dim_store['cluster_zone'].value_counts())

dim_store['store_size_sqft'].describe()

### 3.4 Supplier Overview

In [ ]:
# Supplier type and lead time
print(dim_supplier['supplier_type'].value_counts())
print("\nLead time (days) stats:")
print(dim_supplier['lead_time_days'].describe())

## 4. Fact Tables – Core Business Metrics

### 4.1 Fact Transaction – Sales, Profit, Returns

In [ ]:
# Load fact_transaction (can be large, we sample or aggregate)
# For heavy tables, we compute KPIs directly in SQL
kpi_query = """
SELECT 
    COUNT(DISTINCT transaction_id) AS total_transactions,
    SUM(quantity) AS total_units_sold,
    SUM(total_sale_amount) AS total_revenue,
    SUM(gross_profit) AS total_gross_profit,
    AVG(gross_margin_pct) AS avg_gross_margin_pct,
    SUM(CASE WHEN return_flag = 'Yes' THEN 1 ELSE 0 END) AS return_count
FROM fact_transaction;
"""
kpi_df = pd.read_sql(kpi_query, engine)
display(kpi_df)

# Return rate
return_rate = kpi_df['return_count'].iloc[0] / kpi_df['total_transactions'].iloc[0] * 100
print(f"Return rate: {return_rate:.2f}%")

In [ ]:
# Distribution of numeric fields in transaction lines
numeric_cols = ['quantity', 'discount_pct', 'sale_price', 'total_sale_amount', 'gross_margin_pct']
# Load a sample for performance (e.g., 500k rows). Adjust limit as needed.
sample_trans = pd.read_sql("SELECT quantity, discount_pct, sale_price, total_sale_amount, gross_margin_pct FROM fact_transaction LIMIT 500000", engine)
sample_trans.describe()

In [ ]:
# Outlier detection – boxplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sample_trans.boxplot(column='quantity', ax=axes[0,0])
sample_trans.boxplot(column='discount_pct', ax=axes[0,1])
sample_trans.boxplot(column='total_sale_amount', ax=axes[1,0])
sample_trans.boxplot(column='gross_margin_pct', ax=axes[1,1])
plt.tight_layout()
plt.show()

### 4.2 Fact Inventory – Stock Performance

In [ ]:
# Inventory KPIs
inv_query = """
SELECT 
    AVG(stockout_flag) AS avg_stockout_rate,
    AVG(reorder_triggered) AS avg_reorder_rate,
    AVG(days_of_stock_remaining) AS avg_days_stock,
    SUM(inventory_value_inr) AS total_inventory_value
FROM fact_inventory;
"""
inv_kpi = pd.read_sql(inv_query, engine)
display(inv_kpi)

# Inventory turnover proxy: total units sold / avg closing stock (we need to compute by product-store week)
# This is a complex metric, we can compute later in a separate cell.

## 5. Deeper Business Analysis

### 5.1 Sales by Store & Geography

In [ ]:
sales_store_query = """
SELECT 
    s.store_id,
    s.store_name,
    s.city,
    s.city_tier,
    s.cluster_zone,
    s.store_type,
    SUM(ft.total_sale_amount) AS revenue,
    SUM(ft.gross_profit) AS profit,
    COUNT(DISTINCT ft.transaction_id) AS num_transactions
FROM fact_transaction ft
JOIN dim_store s ON ft.store_id = s.store_id
GROUP BY s.store_id, s.store_name, s.city, s.city_tier, s.cluster_zone, s.store_type
ORDER BY revenue DESC
LIMIT 20;
"""
sales_store = pd.read_sql(sales_store_query, engine)
display(sales_store.head(10))

# Average revenue by city tier
tier_rev = sales_store.groupby('city_tier')['revenue'].mean().reset_index()
tier_rev.plot(x='city_tier', y='revenue', kind='bar', legend=False)
plt.title('Average Revenue per Store by City Tier')
plt.ylabel('Revenue (INR)')
plt.show()

### 5.2 Product & Category Performance

In [ ]:
prod_perf_query = """
SELECT 
    p.category,
    p.sub_category,
    p.product_name,
    SUM(ft.quantity) AS units_sold,
    SUM(ft.total_sale_amount) AS revenue,
    SUM(ft.gross_profit) AS profit,
    AVG(ft.gross_margin_pct) AS avg_margin
FROM fact_transaction ft
JOIN dim_product p ON ft.product_id = p.product_id
GROUP BY p.category, p.sub_category, p.product_name
ORDER BY revenue DESC
LIMIT 15;
"""
top_products = pd.read_sql(prod_perf_query, engine)
display(top_products)

# Top categories by revenue
cat_rev = top_products.groupby('category')['revenue'].sum().sort_values(ascending=False)
cat_rev.plot(kind='bar', title='Revenue by Product Category')
plt.ylabel('Revenue (INR)')
plt.show()

### 5.3 Time Series – Trends & Seasonality

In [ ]:
# Daily sales joined with dim_date
daily_sales_query = """
SELECT 
    d.full_date,
    d.year,
    d.month,
    d.weekday_name,
    d.is_weekend,
    SUM(ft.total_sale_amount) AS daily_revenue,
    COUNT(DISTINCT ft.transaction_id) AS daily_transactions
FROM fact_transaction ft
JOIN dim_date d ON ft.date_id = d.date_id
GROUP BY d.full_date, d.year, d.month, d.weekday_name, d.is_weekend
ORDER BY d.full_date;
"""
daily_sales = pd.read_sql(daily_sales_query, engine)
daily_sales['full_date'] = pd.to_datetime(daily_sales['full_date'])

# Plot daily revenue over time
plt.figure(figsize=(15,5))
plt.plot(daily_sales['full_date'], daily_sales['daily_revenue'], alpha=0.7)
plt.title('Daily Revenue Trend')
plt.xlabel('Date')
plt.ylabel('Revenue (INR)')
plt.show()

# Monthly aggregated
monthly = daily_sales.groupby(['year','month']).agg({'daily_revenue':'sum', 'daily_transactions':'sum'}).reset_index()
monthly['month_start'] = pd.to_datetime(monthly[['year','month']].assign(day=1))
monthly.plot(x='month_start', y='daily_revenue', kind='line', marker='o')
plt.title('Monthly Revenue')
plt.show()

In [ ]:
# Weekday vs weekend performance
weekday_perf = daily_sales.groupby('weekday_name')['daily_revenue'].mean().sort_values()
weekday_perf.plot(kind='bar', title='Average Daily Revenue by Weekday')
plt.ylabel('Avg Revenue (INR)')
plt.show()

### 5.4 Customer Segmentation & Loyalty Impact

In [ ]:
cust_seg_query = """
SELECT 
    c.loyalty_member,
    c.age_group,
    c.gender,
    AVG(ft.total_sale_amount) AS avg_transaction_value,
    COUNT(DISTINCT ft.transaction_id) AS total_transactions,
    SUM(ft.total_sale_amount) AS total_revenue
FROM fact_transaction ft
JOIN dim_customer c ON ft.customer_id = c.customer_id
GROUP BY c.loyalty_member, c.age_group, c.gender
ORDER BY total_revenue DESC;
"""
cust_seg = pd.read_sql(cust_seg_query, engine)
display(cust_seg.head(10))

# Loyalty vs non-loyalty average spend
loyalty_avg = cust_seg.groupby('loyalty_member')['avg_transaction_value'].mean()
loyalty_avg.plot(kind='bar', title='Avg Transaction Value: Loyalty vs Non Loyalty')
plt.ylabel('INR')
plt.show()

### 5.5 Discount & Festive Period Analysis

In [ ]:
discount_query = """
SELECT 
    is_festive_period,
    festival_name,
    AVG(discount_pct) AS avg_discount,
    AVG(quantity) AS avg_quantity,
    AVG(total_sale_amount) AS avg_basket_value,
    COUNT(*) AS num_lines
FROM fact_transaction
GROUP BY is_festive_period, festival_name
ORDER BY avg_basket_value DESC;
"""
discount_effect = pd.read_sql(discount_query, engine)
display(discount_effect)

### 5.6 Inventory Health – Stockouts & Reorder Efficiency

In [ ]:
stockout_query = """
SELECT 
    p.category,
    AVG(fi.stockout_flag) AS stockout_rate,
    AVG(fi.days_of_stock_remaining) AS avg_days_stock,
    SUM(fi.units_sold) AS total_units_sold
FROM fact_inventory fi
JOIN dim_product p ON fi.product_id = p.product_id
GROUP BY p.category
ORDER BY stockout_rate DESC;
"""
stockout_by_cat = pd.read_sql(stockout_query, engine)
display(stockout_by_cat)

plt.figure(figsize=(10,6))
sns.barplot(data=stockout_by_cat, x='stockout_rate', y='category')
plt.title('Stockout Rate by Product Category')
plt.xlabel('Stockout Rate (proportion of weeks)')
plt.show()

### 5.7 Supplier Performance & Lead Time Impact

In [ ]:
supplier_query = """
SELECT 
    s.supplier_name,
    s.supplier_type,
    s.lead_time_days,
    s.is_preferred_vendor,
    SUM(ft.total_sale_amount) AS revenue_generated,
    AVG(ft.gross_margin_pct) AS avg_margin,
    SUM(ft.gross_profit) AS total_profit
FROM fact_transaction ft
JOIN dim_supplier s ON ft.supplier_id = s.supplier_id
GROUP BY s.supplier_name, s.supplier_type, s.lead_time_days, s.is_preferred_vendor
ORDER BY revenue_generated DESC
LIMIT 20;
"""
supplier_perf = pd.read_sql(supplier_query, engine)
display(supplier_perf.head(10))

## 6. Correlation Analysis (Numerical Fields)

In [ ]:
# Use a sample from fact_transaction
corr_sample = pd.read_sql("SELECT quantity, discount_pct, sale_price, total_sale_amount, gross_margin_pct FROM fact_transaction LIMIT 200000", engine)
corr_matrix = corr_sample.corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix – Transaction Line Items')
plt.show()

## 7. Key Insights & Next Steps

Based on the above EDA, typical recommendations could include:
- **Inventory**: Categories with high stockout rates (e.g., perishables) need safety stock review.
- **Pricing/Discounts**: Analyse if high discounts lead to incremental volume or margin erosion.
- **Customer loyalty**: Loyalty members show higher transaction values – invest in retention campaigns.
- **Store expansion**: Stores in city tier 1 generate highest revenue per store; evaluate tier 2/3 potential.
- **Festive periods**: Revenue peaks during festivals – plan promotions and inventory accordingly.
- **Supplier negotiations**: Long lead times correlate with stockouts; consider rebalancing contracts.

**Further analysis**:
- Cohort analysis of customer retention
- Market basket analysis (product affinity)
- Predictive modelling for sales/demand forecasting
- Inventory turnover ratio by store and product family